# Stage 3 -- Theme Allocation: Combined Full Moments

## Input
- `Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_means_theme_assignment.csv` -- the verified means theme assignment, used as the source of truth for base factor → theme mapping
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_full_moments.parquet` -- schema read only, to obtain exact feature column names
- `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_full_moments_factor_inventory.csv` -- source/description/moment_type metadata for each feature

## Purpose
Assigns every feature in the combined full moments table to the same hierarchical theme/subtheme taxonomy defined in Notebook 01. Rather than re-specifying the mapping from scratch, this notebook derives each moment column's theme by stripping its moment suffix to recover the base factor name, then looking up that base factor in the already-verified means assignment. This guarantees that all five moments of a given factor (cwmean, cwstd, cwskew, cwkurt, spread) inherit exactly the same theme and subtheme as their cwmean counterpart in the means table.

---

## Pipeline

### Step 1: Load the Verified Means Assignment
The `combined_means_theme_assignment.csv` from Notebook 01 is loaded and indexed by column name. This serves as the authoritative source of truth: every base factor already has a validated theme assignment.

### Step 2: Read Feature Column Names from Full Moments Table
The parquet schema is read without loading data. `date`, `target_daily_return`, and `target_monthly_return` are excluded.

### Step 3: Load Full Moments Inventory for Metadata
The `combined_full_moments_factor_inventory.csv` is loaded to supply `base_factor`, `moment_type`, `frequency`, `panel`, `source`, and `description` for each column.

### Step 4: Map Each Moment Column to Its Base Factor's Theme
For each feature column, two lookup strategies are tried in order:

**Try 1 -- Direct match:** if the column name exists as-is in the means assignment (macro factors, calendar features, regime indicators -- all of which appear with identical names in both tables), the theme is taken directly.

**Try 2 -- Suffix stripping:** if the column ends with one of the five moment suffixes (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`), the suffix is stripped to recover the base column name (e.g., `bid_ask_spread_cwmean` → `bid_ask_spread`, `monthly_AM_cwmean` → `monthly_AM`). The base name is then looked up in the means assignment. This works for both daily stock moment columns and monthly stock moment columns (which retain the `monthly_` prefix after suffix stripping).

Columns that match neither strategy are marked with `theme_id = -1` and `theme_name = '???'` and added to the unmatched list.

### Step 5: Validate
- **Unmatched count:** must be zero for validation to pass
- **Duplicate check:** confirms no column appears more than once
- **Theme summary table:** features and subthemes per theme, printed for review
- **Moment type breakdown:** count of cwmean, cwstd, cwskew, cwkurt, spread, and raw level columns

### Step 6: Save
Results saved to CSV.

---

## Key Design Decisions
- **Inherits from means assignment rather than re-specifying.** This is the critical design choice: rather than hand-coding theme assignments for ~2,200 moment columns, the notebook derives them automatically from the ~725-column means assignment. This guarantees consistency and eliminates the risk of a factor being assigned to different themes in the two tables.
- **Direct match handles macro/calendar/regime columns** because those appear with identical names in both the means and full moments tables (they are raw level features with no moment suffix).
- **Suffix stripping handles stock moment columns** for both daily (`bid_ask_spread_cwstd`) and monthly (`monthly_AM_cwkurt`) variants.
- **`theme_id = -1` sentinel** marks unmatched columns clearly in the output, making them easy to filter and investigate.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_full_moments_theme_assignment.csv` -- one row per feature, columns: `column`, `base_factor`, `moment_type`, `theme_id`, `theme_name`, `subtheme_id`, `subtheme_name`, `frequency`, `panel`, `source`, `description`

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from collections import Counter

BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD THE VERIFIED MEANS ASSIGNMENT AS THE SOURCE OF TRUTH
# ═══════════════════════════════════════════════════════════════════════════════

means_assignment = pd.read_csv(BASE / 'themes' / 'combined_means_theme_assignment.csv')
means_map = means_assignment.set_index('column')[['theme_id', 'theme_name', 'subtheme_id', 'subtheme_name']].to_dict('index')

print(f"Means assignment loaded: {len(means_map)} base factors")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: READ ACTUAL COLUMN NAMES FROM COMBINED FULL MOMENTS TABLE
# ═══════════════════════════════════════════════════════════════════════════════

schema = pq.read_schema(BASE / 'model_market_combined_full_moments.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return', 'target_monthly_return']]
print(f"Features in combined full moments table: {len(features)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: LOAD THE FULL MOMENTS INVENTORY FOR METADATA
# ═══════════════════════════════════════════════════════════════════════════════

inv = pd.read_csv(BASE / 'combined_full_moments_factor_inventory.csv')
inv_map = inv.set_index('column').to_dict('index')

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: MAP EACH MOMENT COLUMN TO ITS BASE FACTOR'S THEME/SUBTHEME
# ═══════════════════════════════════════════════════════════════════════════════

moment_suffixes = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']

rows = []
matched = 0
unmatched = []

for col in features:
    meta = inv_map.get(col, {})
    base_factor = meta.get('base_factor', '')
    moment_type = meta.get('moment_type', 'raw level')
    
    # Strategy: for stock-level moment columns, strip suffix to find the
    # means-table base factor name. For macro columns, the name is identical.
    
    # Try 1: direct match (macro factors, calendar, regime — identical names)
    if col in means_map:
        info = means_map[col]
        rows.append({
            'column': col,
            'base_factor': base_factor if base_factor else col,
            'moment_type': moment_type,
            'theme_id': info['theme_id'],
            'theme_name': info['theme_name'],
            'subtheme_id': info['subtheme_id'],
            'subtheme_name': info['subtheme_name'],
            'frequency': meta.get('frequency', ''),
            'panel': meta.get('panel', ''),
            'source': meta.get('source', ''),
            'description': meta.get('description', ''),
        })
        matched += 1
        continue
    
    # Try 2: strip moment suffix to recover base factor name for means lookup
    found = False
    for suffix in moment_suffixes:
        if col.endswith(suffix):
            # For daily stock: "bid_ask_spread_cwmean" → "bid_ask_spread"
            # For monthly stock: "monthly_AM_cwmean" → "monthly_AM"
            base_col = col[:-len(suffix)]
            
            if base_col in means_map:
                info = means_map[base_col]
                rows.append({
                    'column': col,
                    'base_factor': base_factor if base_factor else base_col,
                    'moment_type': suffix[1:],  # remove leading underscore
                    'theme_id': info['theme_id'],
                    'theme_name': info['theme_name'],
                    'subtheme_id': info['subtheme_id'],
                    'subtheme_name': info['subtheme_name'],
                    'frequency': meta.get('frequency', ''),
                    'panel': meta.get('panel', ''),
                    'source': meta.get('source', ''),
                    'description': meta.get('description', ''),
                })
                matched += 1
                found = True
                break
    
    if not found:
        rows.append({
            'column': col,
            'base_factor': base_factor if base_factor else col,
            'moment_type': moment_type,
            'theme_id': -1,
            'theme_name': '???',
            'subtheme_id': '???',
            'subtheme_name': '???',
            'frequency': meta.get('frequency', ''),
            'panel': meta.get('panel', ''),
            'source': meta.get('source', ''),
            'description': meta.get('description', ''),
        })
        unmatched.append(col)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("VALIDATION")
print("=" * 90)

print(f"\n  Total features: {len(features)}")
print(f"  Matched: {matched}")
print(f"  Unmatched: {len(unmatched)}")

if unmatched:
    print(f"\n  ⚠ Unmatched columns:")
    for c in unmatched[:20]:
        print(f"    {c}")
    if len(unmatched) > 20:
        print(f"    ... and {len(unmatched) - 20} more")

# Check for duplicates
col_counts = Counter(r['column'] for r in rows)
dupes = {c: n for c, n in col_counts.items() if n > 1}
if dupes:
    print(f"\n  ⚠ Duplicates: {len(dupes)}")
else:
    print(f"  ✓ No duplicate assignments")

# Theme summary
result = pd.DataFrame(rows)
print(f"\n  Theme summary:")
theme_summary = result.groupby(['theme_id', 'theme_name']).agg(
    features=('column', 'count'),
    subthemes=('subtheme_id', 'nunique')
).reset_index()

print(f"  {'#':<4s} {'Theme':<40s} {'Features':>10s} {'Subthemes':>10s}")
print("  " + "-" * 67)
for _, row in theme_summary.iterrows():
    print(f"  {int(row['theme_id']):<4d} {row['theme_name']:<40s} {int(row['features']):>10d} {int(row['subthemes']):>10d}")

print(f"\n  Total: {result.shape[0]} features, {result['subtheme_id'].nunique()} subthemes")

# Moment type breakdown
print(f"\n  Moment type breakdown:")
print(result['moment_type'].value_counts().to_string())

if len(unmatched) == 0 and len(dupes) == 0:
    print(f"\n  ✓ VALIDATION PASSED")
else:
    print(f"\n  ⚠ VALIDATION FAILED — fix issues above")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("SAVING")
print("=" * 90)

out_path = BASE / 'themes' / 'combined_full_moments_theme_assignment.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(out_path, index=False)

print(f"\n  ✓ Saved: {out_path}")
print(f"    {result.shape[0]} rows × {result.shape[1]} columns")
print(f"\n  Sample rows:")
print(result[['column', 'moment_type', 'theme_id', 'theme_name', 'subtheme_id']].head(10).to_string(index=False))

Means assignment loaded: 722 base factors
Features in combined full moments table: 2212

VALIDATION

  Total features: 2212
  Matched: 2212
  Unmatched: 0
  ✓ No duplicate assignments

  Theme summary:
  #    Theme                                      Features  Subthemes
  -------------------------------------------------------------------
  1    Liquidity & Market Quality                      305          9
  2    Order Flow & Participation                      350          9
  3    Volatility & Options                            326         11
  4    Momentum & Reversal                             172          9
  5    Valuation                                       120          5
  6    Profitability & Earnings Quality                120          4
  7    Investment & Corporate Structure                214          8
  8    Analyst Expectations & Sentiment                235          9
  9    Interest Rates & Monetary Policy                 80         10
  10   Credit Conditions    